# S12 — quantum (VQC) classifier for benign vs malignant mammograms, as in the supervisor's poster (CMMD + CBIS-DDSM)
Rules: `S12_preregistration.txt` (v2, fixed before any S12 code ran). Exact quantum-circuit **simulation** (no quantum hardware).
**Poster pipeline:** image → 4×4 patches → PCA + angle encoding → VQC (Rx Ry Rz + CNOT ring, 8 qubits × 4 layers) → ⟨Z⟩ → dense → softmax.
**Arms:** VQC on patches · Hybrid CNN+VQC · fair controls on identical inputs (same-size MLP, logistic regression, RBF-SVM) ·
literature controls (no-entanglement VQC, data re-uploading, quantum-kernel SVM) · poster's other encodings (amplitude, basis) ·
poster baselines Classical CNN / ResNet50 / EfficientNet-B0 / ConvNeXt-Tiny. Primary Q1–Q3 on BOTH datasets.

**Inputs (Add Input):** 1) the **S8 replication** notebook output (`cmmd_data/`, `clean_data/`, `cmmd_clf_s0.pt`) ·
2) the **S2 baseline-audit** notebook output (`a6_clf_s0.pt`, the CBIS-DDSM ConvNeXt; S8 used the same one).
GPU T4 ×2 · Internet ON · Run All (editor) or Save & Run All. **≈ 2–3 h.** Send back `S12_results.zip`.

In [ ]:
# ===================== CELL 1 — code + inputs =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
OUT = "/kaggle/working/S12"; WD = "/kaggle/working/s12_code"; PL = "/kaggle/working/pl_pkgs"
QUANTUM_BUDGET_H = 2.5            # secondary quantum configurations are skipped after this (main ones always run)
os.makedirs(OUT, exist_ok=True); os.makedirs(WD, exist_ok=True)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA", "CMMD")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(os.path.join(os.environ.get("PA_TMP", "/tmp"), f"{split}_cache.npz"))      # S8: dataset switch
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(os.path.join(os.environ.get("PA_DATA", "/kaggle/working/clean_data"), f"{split}_cache.npz"))   # S8: dataset switch
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "qsim.py": r'''# qsim.py - exact state-vector simulator for the poster's Variational Quantum Circuit (VQC), batched, written once for
# both numpy (local tests) and torch (training with autograd). Poster: angle encoding -> L x [Rx Ry Rz per qubit
# (trainable) + entangling CNOT ring] -> measure <Z_i> of every qubit.
import numpy as np

class Backend:
    def __init__(self, lib):
        self.lib = lib; self.torch = lib.__name__ == "torch"
    def cplx(self, re, im):
        return self.lib.complex(re, im) if self.torch else re + 1j * im
    def stack(self, L, dim):
        return self.lib.stack(L, dim) if self.torch else np.stack(L, dim)
    def zeros_like_real(self, x):
        return self.lib.zeros_like(x)
    def index(self, perm):
        return self.lib.as_tensor(perm, dtype=self.lib.long) if self.torch else np.asarray(perm)

def rot(bk, theta, kind):
    """theta: (...,) real -> (...,2,2) complex rotation matrix RX / RY / RZ"""
    L = bk.lib; c, s = L.cos(theta / 2), L.sin(theta / 2); z = bk.zeros_like_real(theta)
    if kind == "x":
        rows = [[bk.cplx(c, z), bk.cplx(z, -s)], [bk.cplx(z, -s), bk.cplx(c, z)]]
    elif kind == "y":
        rows = [[bk.cplx(c, z), bk.cplx(-s, z)], [bk.cplx(s, z), bk.cplx(c, z)]]
    else:
        rows = [[bk.cplx(c, -s), bk.cplx(z, z)], [bk.cplx(z, z), bk.cplx(c, s)]]
    return bk.stack([bk.stack(r, -1) for r in rows], -2)

def apply_1q(bk, state, U, q, n):
    """state (B, 2^n); U (2,2) shared or (B,2,2) per sample; qubit q = most-significant-first (PennyLane wire order)"""
    B = state.shape[0]; s = state.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    s = bk.lib.einsum("ij,akjb->akib", U, s) if U.ndim == 2 else bk.lib.einsum("aij,akjb->akib", U, s)
    return s.reshape(B, 2 ** n)

def cnot_ring_perm(n):
    """basis permutation of CNOT(0,1) CNOT(1,2) ... CNOT(n-1,0) applied in this order (n >= 3; n = 2 -> one CNOT)"""
    pairs = [(i, (i + 1) % n) for i in range(n)] if n > 2 else [(0, 1)]
    idx = np.arange(2 ** n)
    bit = lambda v, q: (v >> (n - 1 - q)) & 1
    for c, t in pairs:                       # new_state[i] = old_state[perm(i)], composed gate by gate
        src = np.array([i ^ (1 << (n - 1 - t)) if bit(i, c) else i for i in range(2 ** n)])
        idx = idx[src]
    return idx

def zsign(n):
    return np.array([[1 - 2 * ((i >> (n - 1 - q)) & 1) for q in range(n)] for i in range(2 ** n)], dtype=np.float32)

def angle_state(bk, angles):
    """product state RY(theta_q)|0> on every wire: (B,n) -> (B,2^n) (exact; identical to applying RY gates)"""
    B, n = angles.shape; L = bk.lib
    st = bk.cplx(L.ones((B, 1)) if bk.torch else np.ones((B, 1)), L.zeros((B, 1)) if bk.torch else np.zeros((B, 1)))
    for q in range(n):
        a = angles[:, q]; v = bk.stack([bk.cplx(L.cos(a / 2), L.zeros_like(a)), bk.cplx(L.sin(a / 2), L.zeros_like(a))], 1)
        st = (st.unsqueeze(-1) * v.unsqueeze(1)).reshape(B, -1) if bk.torch else (st[:, :, None] * v[:, None, :]).reshape(B, -1)
    return st

def vqc_state(bk, angles, W, perm, init=None, reupload=False, entangle=True):
    """angles (B,n): encoding angles; W (L,n,3) trainable (Rx,Ry,Rz) -> final state (B,2^n).
    init      : optional prepared state (B,2^n) (amplitude / basis encoding); default = angle encoding RY(angles)|0>
    reupload  : re-apply the encoding RY(angles) before every layer after the first (data re-uploading)
    entangle  : False removes the CNOT ring (no-entanglement ablation)"""
    n = angles.shape[1]
    st = angle_state(bk, angles) if init is None else init
    for l in range(W.shape[0]):
        if reupload and l > 0:
            Ry = rot(bk, angles, "y")                                                                # (B,n,2,2)
            for q in range(n):
                st = apply_1q(bk, st, Ry[:, q], q, n)
        U = rot(bk, W[l, :, 2], "z") @ rot(bk, W[l, :, 1], "y") @ rot(bk, W[l, :, 0], "x")       # (n,2,2): Rx first, then Ry, Rz
        for q in range(n):
            st = apply_1q(bk, st, U[q], q, n)
        if entangle:
            st = st[:, perm]
    return st

def feature_map_state(bk, angles, reps=2):
    """fixed (non-trainable) entangling feature map for the quantum kernel: reps x [RY(x) on all wires, CNOT ring,
    RZ(x) on all wires] starting from |0...0> (in the spirit of Havlicek et al. 2019)"""
    B, n = angles.shape; perm = cnot_ring_perm(n)
    st = angle_state(bk, angles)
    for r in range(reps):
        if r > 0:
            Ry = rot(bk, angles, "y")
            for q in range(n):
                st = apply_1q(bk, st, Ry[:, q], q, n)
        st = st[:, perm]
        Rz = rot(bk, angles, "z")
        for q in range(n):
            st = apply_1q(bk, st, Rz[:, q], q, n)
    return st

def expz(bk, st, zs):
    p = st.real ** 2 + st.imag ** 2
    return p @ zs

def bloch(bk, st, q, n):
    """Bloch vector (<X>,<Y>,<Z>) of qubit q (reduced state) -> (B,3)"""
    B = st.shape[0]; s = st.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    a0, a1 = s[:, :, 0, :], s[:, :, 1, :]
    r01 = (a0 * a1.conj()).sum((1, 2)) if bk.torch else (a0 * a1.conj()).sum((1, 2))   # rho_01 = sum psi_0 psi_1*
    pz = (abs(a0) ** 2).sum((1, 2)) - (abs(a1) ** 2).sum((1, 2))
    return bk.stack([2 * r01.real, -2 * r01.imag, pz], 1)

def n_params(n, L):
    return 3 * n * L
''',
         "s12_gpu.py": r'''# s12_gpu.py - S12 GPU part (rules: S12_preregistration.txt).
#   --task feats : frozen ConvNeXt-Tiny of the dataset (CMMD: cmmd_clf_s0.pt from S8; CBIS-DDSM: a6_clf_s0.pt from S2)
#                  -> 768-d penultimate features of train-fit / val / test
#                  (the "CNN" of the poster's Hybrid CNN+VQC; same layers as the KID-mammo features)
#   --task cnn   : the poster's classical baselines, fine-tuned end to end with the fixed S2 recipe:
#                  smallcnn (4-block CNN from scratch, lr 1e-3), resnet50, efficientnet_b0, convnext_tiny (lr 1e-4)
import os, sys, json, math, time, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from a6_classifier import build, prep, augment, predict, auc, dev

ap = argparse.ArgumentParser()
ap.add_argument("--task", choices=["feats", "cnn"], required=True)
ap.add_argument("--ds", choices=["cmmd", "cbis"], required=True)
ap.add_argument("--data", required=True)                 # cmmd_data/ or clean_data/ folder
ap.add_argument("--clf", default="")                     # cmmd_clf_s0.pt (CMMD) or a6_clf_s0.pt (CBIS-DDSM)
ap.add_argument("--archs", default="smallcnn,resnet50,efficientnet_b0,convnext_tiny")
ap.add_argument("--seeds", default="0,1,2")
ap.add_argument("--epochs", type=int, default=15)
ap.add_argument("--out", required=True)
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
def log(*m): print(f"[S12-{a.task}-{a.ds}]", *m, flush=True)

def val_mask(patients, frac=0.15, seed=0):
    ups = sorted(set(patients)); rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])
d = np.load(os.path.join(a.data, "train_cache.npz")); xtr, ytr, ptr = d["x"], d["y"].astype(np.int64), d["patient"].astype(str)
vm = val_mask(ptr); XF, YF, XV, YV = xtr[~vm], ytr[~vm], xtr[vm], ytr[vm]
t = np.load(os.path.join(a.data, "test_cache.npz")); XT, YT = t["x"], t["y"].astype(np.int64)
if a.smoke:
    XF, YF, XV, YV, XT, YT, a.epochs = XF[:64], YF[:64], XV[:32], YV[:32], XT[:40], YT[:40], 1

if a.task == "feats":
    m = build(); m.load_state_dict({k: v.float() for k, v in torch.load(a.clf, map_location="cpu").items()}); m = m.to(dev).eval()
    @torch.no_grad()
    def feats(X, bs=50):
        out = []
        for i in range(0, len(X), bs):
            xb = torch.from_numpy(X[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
            with torch.autocast("cuda", dtype=torch.float16):
                out.append(m.classifier[1](m.classifier[0](m.avgpool(m.features(prep(xb))))).float().cpu())
        return torch.cat(out).numpy()
    P = {"f_train": feats(XF), "f_val": feats(XV), "f_test": feats(XT), "y_train": YF, "y_val": YV, "y_test": YT,
         "p_val": predict(m, XV), "p_test": predict(m, XT)}
    np.savez(os.path.join(a.out, f"cnn_features_{a.ds}.npz"), **P)
    log(f"features train {P['f_train'].shape} val {P['f_val'].shape} test {P['f_test'].shape} | frozen classifier itself: "
        f"val AUC {auc(P['p_val'], YV):.3f} test AUC {auc(P['p_test'], YT):.3f}")
    raise SystemExit(0)

class SmallCNN(nn.Module):                                 # the poster's "Classical CNN"
    def __init__(self):
        super().__init__(); L = []; c = 3
        for o in (32, 64, 128, 256):
            L += [nn.Conv2d(c, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.Conv2d(o, o, 3, padding=1),
                  nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2)]; c = o
        self.f = nn.Sequential(*L); self.h = nn.Linear(256, 2)
    def forward(self, x):
        return self.h(F.adaptive_avg_pool2d(self.f(x), 1).flatten(1))
def make(arch):
    import torchvision
    if arch == "smallcnn":
        return SmallCNN(), 1e-3
    if arch == "resnet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2"); m.fc = nn.Linear(m.fc.in_features, 2); return m, 1e-4
    if arch == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1"); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2); return m, 1e-4
    return build(), 1e-4

def train(arch, seed, epochs, bs=32):                      # = a6_classifier.train_clf with a selectable architecture
    Xg, Yg = torch.from_numpy(XF).to(dev), torch.from_numpy(YF).long().to(dev)
    w = torch.tensor([len(YF) / (2 * max(1, (YF == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed); m, lr = make(arch); m = m.to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.05)
    spe = math.ceil(len(YF) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda"); g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(YF), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev); tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss.detach())
        if (ep + 1) % 5 == 0 or ep == 0:
            log(f"{arch} seed {seed} epoch {ep + 1}/{epochs} loss {tl / spe:.4f}")
    return m

for arch in a.archs.split(","):
    for seed in [int(s) for s in a.seeds.split(",")]:
        f = os.path.join(a.out, f"pred_{a.ds}_cnn_{arch}_s{seed}.npz")
        if os.path.exists(f):
            log(f"{f} exists -> skipped"); continue
        t0 = time.time(); m = train(arch, seed, a.epochs).eval()
        pv, pt = predict(m, XV), predict(m, XT); np.savez(f, val=pv, test=pt, y_val=YV, y_test=YT)
        log(f"{arch} seed {seed}: params {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M | val AUC {auc(pv, YV):.3f} | "
            f"test AUC {auc(pt, YT):.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
''',
         "s12_quantum.py": r'''# s12_quantum.py - S12 quantum part for ONE dataset (CPU, exact state-vector simulation; rules: S12_preregistration.txt).
# Poster pipeline: image -> preprocessing & patch extraction -> encoding (PCA + angle = main; amplitude / basis = poster's
# other schemes) -> VQC (Rx Ry Rz + CNOT ring) x L -> measure <Z_i> -> classical dense layer -> softmax.
#   "pure"  : VQC on 4x4 patch statistics of the image (poster "VQC (Ours)")
#   "hybrid": VQC on frozen ConvNeXt features (poster "Hybrid CNN+VQC")
# Fair controls on IDENTICAL inputs: logistic regression; MLP with the SAME number of trainable parameters as the VQC head.
# Literature controls: no-entanglement VQC (Bowles et al. 2024), data re-uploading (Perez-Salinas et al. 2020),
# quantum-kernel SVM (Havlicek et al. 2019) vs classical RBF-kernel SVM.
import os, sys, json, time, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import qsim as Q

ap = argparse.ArgumentParser()
ap.add_argument("--ds", choices=["cmmd", "cbis"], required=True)
ap.add_argument("--data", required=True); ap.add_argument("--feats", required=True); ap.add_argument("--out", required=True)
ap.add_argument("--seeds", default="0,1,2,3,4"); ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--budget_h", type=float, default=3.0)        # secondary configurations are skipped once this is used up
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
torch.set_num_threads(2); T0 = time.time()
def log(*m): print(f"[S12-quantum-{a.ds}]", *m, flush=True)
bk = Q.Backend(torch)

def val_mask(patients, frac=0.15, seed=0):
    ups = sorted(set(patients)); rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])
d = np.load(os.path.join(a.data, "train_cache.npz")); vm = val_mask(d["patient"].astype(str))
X = {"train": d["x"][~vm], "val": d["x"][vm], "test": np.load(os.path.join(a.data, "test_cache.npz"))["x"]}
C = np.load(a.feats); Y = {k: C[f"y_{k}"].astype(np.int64) for k in ("train", "val", "test")}
if a.smoke:
    X = {k: v[:len(Y[k])] for k, v in X.items()}
assert all(len(X[k]) == len(Y[k]) for k in X), {k: (len(X[k]), len(Y[k])) for k in X}

def patch_feats(U8):
    """poster step 2: resize 256 -> 64, per-image min-max, 4x4 grid of 16x16 patches, per patch (mean, std, high-pass energy)"""
    x = U8.astype(np.float32).reshape(len(U8), 64, 4, 64, 4).mean((2, 4))
    mn = x.min((1, 2), keepdims=True); mx = x.max((1, 2), keepdims=True); x = (x - mn) / (mx - mn + 1e-8)
    pad = np.pad(x, ((0, 0), (1, 1), (1, 1)), mode="edge")
    box = sum(pad[:, i:i + 64, j:j + 64] for i in range(3) for j in range(3)) / 9.0; hp = (x - box) ** 2
    P = lambda z: z.reshape(len(z), 4, 16, 4, 16).transpose(0, 1, 3, 2, 4).reshape(len(z), 16, 256)
    return np.concatenate([P(x).mean(2), P(x).std(2), P(hp).mean(2)], 1)          # (N, 48)
RAW = {"pure": {k: patch_feats(v) for k, v in X.items()}, "hybrid": {k: C[f"f_{k}"] for k in X}}

def pca(kind, n):
    """standardise (train-fit stats) -> PCA to n components (fitted on train-fit) -> z-score"""
    R = RAW[kind]; mu, sd = R["train"].mean(0), R["train"].std(0) + 1e-8
    Z = {k: (v - mu) / sd for k, v in R.items()}
    _, S, Vt = np.linalg.svd(Z["train"], full_matrices=False); V = Vt[:n].T
    P = {k: v @ V for k, v in Z.items()}; m2, s2 = P["train"].mean(0), P["train"].std(0) + 1e-8
    return {k: ((v - m2) / s2).astype(np.float32) for k, v in P.items()}, float((S[:n] ** 2).sum() / (S ** 2).sum())
def encode(kind, n, enc):
    """returns (classical input Z, quantum input) ; angle: 2*arctan(z) | amplitude: 2^n PCA numbers, unit norm, on n qubits |
    basis: sign(z) bits -> computational basis state"""
    if enc == "amp":
        Z, ev = pca(kind, 2 ** n)
        # fix 2026-10-23: patch features have only 48 numbers < 2^6 = 64 amplitudes -> zero-pad (standard amplitude encoding)
        pad = lambda v: np.pad(v, ((0, 0), (0, 2 ** n - v.shape[1]))) if v.shape[1] < 2 ** n else v
        A = {k: pad(v) / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8) for k, v in Z.items()}
        return Z, {k: v.astype(np.float32) for k, v in A.items()}, ev
    Z, ev = pca(kind, n)
    if enc == "basis":
        w = 2 ** np.arange(n - 1, -1, -1)
        return Z, {k: ((v > 0).astype(np.int64) * w).sum(1) for k, v in Z.items()}, ev
    return Z, {k: (2 * np.arctan(v)).astype(np.float32) for k, v in Z.items()}, ev

class VQCHead(nn.Module):
    def __init__(self, n, L, enc="angle", reupload=False, entangle=True):
        super().__init__(); self.n, self.enc, self.reupload, self.entangle = n, enc, reupload, entangle
        self.W = nn.Parameter(0.1 * torch.randn(L, n, 3))                          # near-identity start (pre-stated)
        self.dense = nn.Linear(n, 2); self.perm = Q.cnot_ring_perm(n); self.zs = torch.tensor(Q.zsign(n))
    def state(self, xin):
        B = xin.shape[0]
        if self.enc == "amp":
            init = torch.complex(xin, torch.zeros_like(xin)); ang = torch.zeros(B, self.n)
        elif self.enc == "basis":
            re = torch.zeros(B, 2 ** self.n); re[torch.arange(B), xin] = 1.0
            init = torch.complex(re, torch.zeros_like(re)); ang = torch.zeros(B, self.n)
        else:
            init, ang = None, xin
        return Q.vqc_state(bk, ang, self.W, self.perm, init=init, reupload=self.reupload, entangle=self.entangle)
    def forward(self, xin):
        return self.dense(Q.expz(bk, self.state(xin), self.zs))
class MLPHead(nn.Module):
    def __init__(self, n_in, h):
        super().__init__(); self.net = nn.Sequential(nn.Linear(n_in, h), nn.Tanh(), nn.Linear(h, 2))
    def forward(self, z):
        return self.net(z)
nparams = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

def fit(model, Xtr, ytr, seed, epochs, bs=64, lr=0.01):
    w = torch.tensor([len(ytr) / (2 * max(1, (ytr == c).sum())) for c in (0, 1)], dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr); Xt, Yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)
    g = torch.Generator().manual_seed(seed); hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(Yt), generator=g); tl = 0.0
        for i in range(0, len(Yt), bs):
            idx = perm[i:i + bs]; loss = F.cross_entropy(model(Xt[idx]), Yt[idx], weight=w)
            opt.zero_grad(); loss.backward(); opt.step(); tl += float(loss) * len(idx)
        hist.append(tl / len(Yt))
    return model, hist
@torch.no_grad()
def proba(model, Xn):
    return torch.softmax(model(torch.from_numpy(Xn)), 1)[:, 1].numpy()
def auc(s, y):
    from scipy.stats import rankdata
    r = rankdata(s); n1, n0 = (y == 1).sum(), (y == 0).sum(); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def save(name, pv, pt, **kw):
    np.savez(os.path.join(a.out, f"pred_{a.ds}_{name}.npz"), val=pv, test=pt, y_val=Y["val"], y_test=Y["test"], **kw)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
# (feature set, qubits, layers, encoding, re-upload, entangle, role) - in priority order
CONFIGS = [("hybrid", 8, 4, "angle", False, True, "main"), ("pure", 8, 4, "angle", False, True, "main"),
           ("hybrid", 8, 4, "angle", False, False, "noent"), ("pure", 8, 4, "angle", False, False, "noent"),
           ("hybrid", 8, 4, "angle", True, True, "reupload"), ("pure", 8, 4, "angle", True, True, "reupload"),
           ("hybrid", 6, 4, "amp", False, True, "encoding"), ("pure", 6, 4, "amp", False, True, "encoding"),
           ("hybrid", 8, 4, "basis", False, True, "encoding"),
           ("hybrid", 4, 2, "angle", False, True, "size"), ("hybrid", 4, 4, "angle", False, True, "size"),
           ("hybrid", 8, 2, "angle", False, True, "size")]
if a.smoke:
    CONFIGS = [CONFIGS[0], CONFIGS[1], CONFIGS[4], CONFIGS[6], CONFIGS[8]]; a.epochs = 2
SEEDS = [int(s) for s in a.seeds.split(",")]; META = {}
mpath = os.path.join(a.out, f"quantum_meta_{a.ds}.json")
if os.path.exists(mpath):
    META = json.load(open(mpath))

# ---------------- quantum kernel SVM vs classical RBF SVM (fixed feature map, no training of the circuit) ----------------
for kind in ("hybrid", "pure"):
    Z, ANG, _ = encode(kind, 8, "angle")
    with torch.no_grad():
        S = {k: Q.feature_map_state(Q.Backend(np), v.astype(np.float64), reps=2) for k, v in ANG.items()}
    Ktr = np.abs(S["train"].conj() @ S["train"].T) ** 2
    qs = SVC(kernel="precomputed", C=1.0, class_weight="balanced").fit(Ktr, Y["train"])
    sig = lambda z: 1 / (1 + np.exp(-z))
    pv = sig(qs.decision_function(np.abs(S["val"].conj() @ S["train"].T) ** 2))
    pt = sig(qs.decision_function(np.abs(S["test"].conj() @ S["train"].T) ** 2)); save(f"{kind}_qkernel_s0", pv, pt)
    rb = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced").fit(Z["train"], Y["train"])
    save(f"{kind}_rbfsvm_s0", sig(rb.decision_function(Z["val"])), sig(rb.decision_function(Z["test"])))
    log(f"{kind}: quantum-kernel SVM test AUC {auc(pt, Y['test']):.3f} | RBF SVM test AUC {auc(sig(rb.decision_function(Z['test'])), Y['test']):.3f}")

for kind, n, L, enc, reup, ent, role in CONFIGS:
    if role != "main" and (time.time() - T0) / 3600 > a.budget_h:
        log(f"time budget {a.budget_h} h used -> {role} config {kind} {n}q {L}l {enc} skipped"); continue
    cfg = f"{kind}_q{n}_l{L}_{enc}" + ("_reup" if reup else "") + ("" if ent else "_noent")
    Z, QIN, ev = encode(kind, n, enc)
    vq_params = Q.n_params(n, L) + 2 * n + 2; n_in = Z["train"].shape[1]
    h = max(1, round((vq_params - 2) / (n_in + 3)))                                  # MLP n_in -> h -> 2 with ~ the same parameters
    META[cfg] = {**META.get(cfg, {}), "kind": kind, "qubits": n, "layers": L, "encoding": enc, "reupload": reup, "entangle": ent, "role": role,
                 "pca_components": n_in, "pca_explained_variance": ev, "mlp_hidden": h}
    lr_ = LogisticRegression(class_weight="balanced", max_iter=5000).fit(Z["train"], Y["train"])
    save(f"{cfg}_LR_s0", lr_.predict_proba(Z["val"])[:, 1], lr_.predict_proba(Z["test"])[:, 1])
    for seed in SEEDS:
        for head in ("VQC", "MLP"):
            f = os.path.join(a.out, f"pred_{a.ds}_{cfg}_{head}_s{seed}.npz")
            if os.path.exists(f):
                continue
            torch.manual_seed(seed); t0 = time.time()
            m = VQCHead(n, L, enc, reup, ent) if head == "VQC" else MLPHead(n_in, h)
            Xin = QIN if head == "VQC" else Z
            m, hist = fit(m, Xin["train"], Y["train"], seed, a.epochs)
            pv, pt = proba(m, Xin["val"]), proba(m, Xin["test"]); META[cfg][f"params_{head}"] = nparams(m)
            extra = {}
            if head == "VQC" and role == "main" and seed == SEEDS[0]:
                with torch.no_grad():
                    st = m.state(torch.from_numpy(Xin["test"]))
                    extra = {"zvec_test": Q.expz(bk, st, m.zs).numpy(), "pca_test": Z["test"],
                             "bloch_test": np.stack([Q.bloch(bk, st, q, n).numpy() for q in range(n)], 1), "W": m.W.detach().numpy()}
            save(f"{cfg}_{head}_s{seed}", pv, pt, loss=np.array(hist), **extra)
            log(f"{cfg} {head} seed {seed}: {nparams(m)} params | val AUC {auc(pv, Y['val']):.3f} | test AUC {auc(pt, Y['test']):.3f} | "
                f"final loss {hist[-1]:.4f} | {time.time() - t0:.0f} s | elapsed {(time.time() - T0) / 60:.0f} min")
    json.dump(META, open(mpath, "w"), indent=1)
log(f"done in {(time.time() - T0) / 60:.0f} min")
''',
         "s12_plcheck.py": r'''# s12_plcheck.py - cross-check of qsim.py (numpy backend) against PennyLane default.qubit (run in its own process).
import sys, os, json, numpy as np
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import qsim as Q
out = sys.argv[1]
import pennylane as qml
res = []
for n, L in ((4, 3), (8, 4)):
    r = np.random.default_rng(n); ang = r.uniform(-3, 3, (5, n)); W = r.uniform(-3, 3, (L, n, 3))
    dv = qml.device("default.qubit", wires=n)
    @qml.qnode(dv)
    def circ(x):
        for q in range(n): qml.RY(x[q], wires=q)
        for l in range(L):
            for q in range(n):
                qml.RX(W[l, q, 0], wires=q); qml.RY(W[l, q, 1], wires=q); qml.RZ(W[l, q, 2], wires=q)
            for q in range(n): qml.CNOT(wires=[q, (q + 1) % n])
        return [qml.expval(qml.PauliZ(q)) for q in range(n)]
    ref = np.array([np.array(circ(x), dtype=float) for x in ang])
    mine = Q.expz(None, Q.vqc_state(Q.Backend(np), ang, W, Q.cnot_ring_perm(n)), Q.zsign(n))
    res.append(float(np.abs(ref - mine).max()))
err = max(res); print(f"PennyLane {qml.__version__} cross-check (4 and 8 qubits): max |<Z> difference| = {err:.2e}", flush=True)
json.dump({"pennylane": qml.__version__, "max_abs_diff": err}, open(os.path.join(out, "pennylane_check.json"), "w"))
sys.exit(0 if err < 1e-5 else 3)
''',
         "s12_stats.py": r'''# s12_stats.py - S12 endpoints, poster-style results tables and figures for CMMD and CBIS-DDSM (CPU).
# usage: python s12_stats.py <S12 output folder> <cmmd_data folder>
import os, sys, glob, json
import numpy as np
from scipy.stats import rankdata
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

D, DATA = sys.argv[1], sys.argv[2]
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e34948", "#c79a1e", "#3aa6b9", "#8a8a8a", "#b0559b", "#4d7f2e"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "font.size": 9, "savefig.dpi": 170, "savefig.bbox": "tight"})
DSN = {"cmmd": "CMMD", "cbis": "CBIS-DDSM"}
def auc(p, y):
    r = rankdata(p); n1 = (y == 1).sum(); n0 = (y == 0).sum(); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def runs(ds, pattern):
    return [dict(np.load(f)) for f in sorted(glob.glob(os.path.join(D, f"pred_{ds}_{pattern}")))]
ens = lambda rs, s: np.mean([r[s] for r in rs], 0)
def metrics(p, y):
    yh = (p >= 0.5).astype(int); tp = ((yh == 1) & (y == 1)).sum(); fp = ((yh == 1) & (y == 0)).sum(); fn = ((yh == 0) & (y == 1)).sum()
    pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn)
    return {"accuracy": float((yh == y).mean()), "precision": float(pr), "recall": float(rc), "f1": float(2 * pr * rc / max(1e-9, pr + rc))}
M8 = "q8_l4_angle"
ARMS = {  # label -> pattern; poster box 9 models + fair controls + literature controls
    "Classical CNN (small, scratch)": "cnn_smallcnn_s*.npz", "ResNet50": "cnn_resnet50_s*.npz",
    "EfficientNet-B0": "cnn_efficientnet_b0_s*.npz", "ConvNeXt-Tiny": "cnn_convnext_tiny_s*.npz",
    "CNN-PCA8 + logistic regression": f"hybrid_{M8}_LR_s*.npz", "CNN-PCA8 + MLP (matched size)": f"hybrid_{M8}_MLP_s*.npz",
    "CNN-PCA8 + RBF-kernel SVM": "hybrid_rbfsvm_s*.npz", "CNN-PCA8 + quantum-kernel SVM": "hybrid_qkernel_s*.npz",
    "Hybrid CNN+VQC": f"hybrid_{M8}_VQC_s*.npz", "Hybrid CNN+VQC, no entanglement": f"hybrid_{M8}_noent_VQC_s*.npz",
    "Hybrid CNN+VQC, data re-uploading": f"hybrid_{M8}_reup_VQC_s*.npz",
    "patch-PCA8 + logistic regression": f"pure_{M8}_LR_s*.npz", "patch-PCA8 + MLP (matched size)": f"pure_{M8}_MLP_s*.npz",
    "patch-PCA8 + RBF-kernel SVM": "pure_rbfsvm_s*.npz", "patch-PCA8 + quantum-kernel SVM": "pure_qkernel_s*.npz",
    "VQC on patches": f"pure_{M8}_VQC_s*.npz", "VQC on patches, no entanglement": f"pure_{M8}_noent_VQC_s*.npz",
    "VQC on patches, data re-uploading": f"pure_{M8}_reup_VQC_s*.npz"}
L = ["S12 REPORT - quantum (VQC) classifier as in the supervisor's poster, benign vs malignant, CMMD + CBIS-DDSM",
     "(rules: S12_preregistration.txt)", "=" * 120,
     "Simulation: exact state-vector simulator (verified against dense matrices; PennyLane cross-check below). No quantum hardware.",
     "8 qubits x 4 layers = 96 circuit + 18 dense parameters; VQC = 5 seeds, MLP = 5 seeds, CNNs = 3 seeds, LR / SVMs deterministic."]
pc = os.path.join(D, "pennylane_check.json")
if os.path.exists(pc):
    j = json.load(open(pc)); L.append(f"PennyLane {j['pennylane']} cross-check: max |<Z> difference| {j['max_abs_diff']:.1e}")
rng = np.random.default_rng(0)
def boot(pa, pb, y, alpha, n=2000):
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]; d = []
    for _ in range(n):
        b = np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]); d.append(auc(pa[b], y[b]) - auc(pb[b], y[b]))
    return float(np.quantile(d, alpha / 2)), float(np.quantile(d, 1 - alpha / 2))
OUT = {}

def do_dataset(ds):
    R = {k: runs(ds, v) for k, v in ARMS.items()}; O = {"arms": {}, "endpoints": {}}
    META = json.load(open(os.path.join(D, f"quantum_meta_{ds}.json"))) if os.path.exists(os.path.join(D, f"quantum_meta_{ds}.json")) else {}
    miss = [k for k, v in R.items() if not v]
    L.extend(["", "#" * 120, f"DATASET: {DSN[ds]}" + (f"   (missing arms: {miss})" if miss else "   (all arms present)"), "#" * 120,
              f"RESULTS TABLE - {DSN[ds]} TEST (ensemble = mean probability over seeds; threshold 0.5 for accuracy / precision / recall / F1)",
              f"  {'model':40s} {'params':>9s} {'seeds':>5s} {'AUC':>6s} {'seed AUC mean+-SD':>18s} {'acc':>6s} {'prec':>6s} {'rec':>6s} {'F1':>6s}"])
    par = {"Hybrid CNN+VQC": META.get(f"hybrid_{M8}", {}).get("params_VQC"), "CNN-PCA8 + MLP (matched size)": META.get(f"hybrid_{M8}", {}).get("params_MLP"),
           "VQC on patches": META.get(f"pure_{M8}", {}).get("params_VQC"), "patch-PCA8 + MLP (matched size)": META.get(f"pure_{M8}", {}).get("params_MLP"),
           "CNN-PCA8 + logistic regression": 9, "patch-PCA8 + logistic regression": 9}
    for k, rs in R.items():
        if not rs:
            continue
        y = rs[0]["y_test"]; p = ens(rs, "test"); s = [auc(r["test"], y) for r in rs]; m = metrics(p, y)
        O["arms"][k] = {"auc": auc(p, y), "seed_aucs": s, "val_auc": auc(ens(rs, "val"), rs[0]["y_val"]), **m}
        sd = f"{np.mean(s):.3f}+-{np.std(s, ddof=1):.3f}" if len(s) > 1 else f"{s[0]:.3f}"
        pp = par.get(k) or ("full net" if k in ("Classical CNN (small, scratch)", "ResNet50", "EfficientNet-B0", "ConvNeXt-Tiny") else "-")
        L.append(f"  {k:40s} {str(pp):>9s} "
                 f"{len(rs):>5d} {auc(p, y):6.3f} {sd:>18s} {m['accuracy']:6.3f} {m['precision']:6.3f} {m['recall']:6.3f} {m['f1']:6.3f}")
    cf = os.path.join(D, f"cnn_features_{ds}.npz")
    if os.path.exists(cf):
        C = np.load(cf); L.append(f"  (frozen ConvNeXt that supplies the hybrid features, its own head: test AUC {auc(C['p_test'], C['y_test']):.3f})")
    def endpoint(name, A, B, alpha, primary):
        if not (R.get(A) and R.get(B)):
            L.append(f"  {name}: arm missing"); return None
        y = R[A][0]["y_test"]; pa, pb = ens(R[A], "test"), ens(R[B], "test"); dd = auc(pa, y) - auc(pb, y); lo, hi = boot(pa, pb, y, alpha)
        eq = (lo > -0.02) and (hi < 0.02)
        verdict = ("PASS" if lo > 0 else "fail") if primary else ("CI excludes 0" if (lo > 0 or hi < 0) else "CI includes 0")
        L.append(f"  {name:4s} AUC[{A}] - AUC[{B}] = {dd:+.4f}  CI{100 * (1 - alpha):.2f}% {lo:+.4f}..{hi:+.4f} -> {verdict}"
                 + (" | within +-0.02 (practically equivalent)" if eq else ""))
        return {"diff": dd, "ci": [lo, hi], "verdict": verdict, "passed": bool(lo > 0), "equivalent_0.02": bool(eq)}
    L.extend(["", f"PRIMARY ENDPOINTS on {DSN[ds]} (98.33 % CI = Bonferroni over Q1-Q3; PASS = CI lower bound > 0)"])
    for q, A, B in (("Q1", "Hybrid CNN+VQC", "CNN-PCA8 + MLP (matched size)"), ("Q2", "VQC on patches", "patch-PCA8 + MLP (matched size)"),
                    ("Q3", "Hybrid CNN+VQC", "ConvNeXt-Tiny")):
        O["endpoints"][q] = endpoint(q, A, B, 0.05 / 3, True)
    L.extend(["", "SECONDARY (95 % CI, descriptive)"])
    for nm, A, B in (("S1", "Hybrid CNN+VQC", "CNN-PCA8 + logistic regression"), ("S2", "VQC on patches", "patch-PCA8 + logistic regression"),
                     ("S3", "Hybrid CNN+VQC", "ResNet50"), ("S4", "Hybrid CNN+VQC", "EfficientNet-B0"),
                     ("S5", "Hybrid CNN+VQC", "Classical CNN (small, scratch)"), ("S6", "Hybrid CNN+VQC", "VQC on patches"),
                     ("S7", "Hybrid CNN+VQC", "Hybrid CNN+VQC, no entanglement"), ("S8", "VQC on patches", "VQC on patches, no entanglement"),
                     ("S9", "Hybrid CNN+VQC, data re-uploading", "CNN-PCA8 + MLP (matched size)"),
                     ("S10", "VQC on patches, data re-uploading", "patch-PCA8 + MLP (matched size)"),
                     ("S11", "CNN-PCA8 + quantum-kernel SVM", "CNN-PCA8 + RBF-kernel SVM"),
                     ("S12", "patch-PCA8 + quantum-kernel SVM", "patch-PCA8 + RBF-kernel SVM")):
        O["endpoints"][nm] = endpoint(nm, A, B, 0.05, False)
    L.extend(["", "ENCODINGS AND CIRCUIT SIZE (descriptive): each VQC vs its size-matched MLP and logistic regression on identical inputs"])
    for cfg, mt in META.items():
        v, m_, lr_ = runs(ds, f"{cfg}_VQC_s*.npz"), runs(ds, f"{cfg}_MLP_s*.npz"), runs(ds, f"{cfg}_LR_s*.npz")
        if v and m_ and lr_:
            y = v[0]["y_test"]
            L.append(f"  {cfg:28s} ({mt['role']:8s}; {mt['pca_components']:2d} PCA numbers keep {100 * mt['pca_explained_variance']:4.1f} % variance): "
                     f"VQC {auc(ens(v, 'test'), y):.3f} ({mt.get('params_VQC')} p) | MLP {auc(ens(m_, 'test'), y):.3f} ({mt.get('params_MLP')} p) | "
                     f"LR {auc(ens(lr_, 'test'), y):.3f}")
    # ---- figures: box 4 (Bloch), box 7 (feature space), box 9 (ROC + AUC bars) ----
    vq = runs(ds, f"hybrid_{M8}_VQC_s0.npz")
    if vq and "bloch_test" in vq[0]:
        v = vq[0]; yb = v["y_test"]; B = v["bloch_test"]
        fig = plt.figure(figsize=(11, 3.8))
        for j, q in enumerate((0, 1, 2)):
            a_ = fig.add_subplot(1, 3, j + 1, projection="3d"); u, w = np.mgrid[0:2 * np.pi:30j, 0:np.pi:15j]
            a_.plot_wireframe(np.cos(u) * np.sin(w), np.sin(u) * np.sin(w), np.cos(w), color="#dddddd", lw=0.3)
            for c, col, lab in ((0, PAL[0], "benign"), (1, PAL[4], "malignant")):
                P = B[yb == c, q]; a_.scatter(P[:, 0], P[:, 1], P[:, 2], s=5, color=col, label=lab, alpha=0.6)
            sep = np.linalg.norm(B[yb == 1, q].mean(0) - B[yb == 0, q].mean(0))
            a_.set_title(f"{DSN[ds]} qubit {q} Bloch vectors\n(class-mean distance {sep:.2f})", fontsize=8); a_.set_box_aspect((1, 1, 1))
            if j == 0:
                a_.legend(fontsize=7, loc="upper left")
        fig.savefig(f"{D}/S12_{ds}_box4_bloch.png"); plt.close(fig)
        L.extend(["", "BLOCH VECTORS (hybrid VQC, seed 0, test images): distance between class-mean Bloch vectors per qubit (max possible 2)",
                  "  " + " | ".join(f"q{q} {np.linalg.norm(B[yb == 1, q].mean(0) - B[yb == 0, q].mean(0)):.3f}" for q in range(B.shape[1]))])
        from sklearn.metrics import silhouette_score
        from sklearn.manifold import TSNE
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.7)); sil = {}
        for a_, (nm, F_) in zip(ax[:2], (("classical PCA input (8 numbers)", v["pca_test"]), ("quantum features <Z> (8 numbers)", v["zvec_test"]))):
            Fc = F_ - F_.mean(0); _, _, Vt_ = np.linalg.svd(Fc, full_matrices=False); P2 = Fc @ Vt_[:2].T; sil[nm] = float(silhouette_score(F_, yb))
            for c, col, lab, mk in ((0, PAL[0], "benign", "o"), (1, PAL[4], "malignant", "^")):
                a_.scatter(P2[yb == c, 0], P2[yb == c, 1], s=9, color=col, marker=mk, label=lab, alpha=0.7)
            a_.set_title(f"{DSN[ds]}: PCA of {nm}\nsilhouette {sil[nm]:+.3f}", loc="left", fontsize=8.5); a_.set_xlabel("PC 1"); a_.set_ylabel("PC 2")
        ax[0].legend(frameon=False, fontsize=7.5)
        T = TSNE(2, random_state=0, perplexity=30, init="pca").fit_transform(v["zvec_test"])
        for c, col, lab, mk in ((0, PAL[0], "benign", "o"), (1, PAL[4], "malignant", "^")):
            ax[2].scatter(T[yb == c, 0], T[yb == c, 1], s=9, color=col, marker=mk, label=lab, alpha=0.7)
        ax[2].set_title(f"{DSN[ds]}: t-SNE of quantum features <Z>", loc="left", fontsize=8.5)
        fig.savefig(f"{D}/S12_{ds}_box7_feature_space.png"); plt.close(fig)
        L.extend(["", "FEATURE-SPACE SEPARATION (silhouette score, test images; higher = classes more separated; -1..1)"]
                 + [f"  {k:34s}: {s:+.3f}" for k, s in sil.items()]); O["silhouette"] = sil
    names = [k for k in ARMS if k in O["arms"]]
    if names:
        fig, ax = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1, 1.1]}); fig.subplots_adjust(wspace=0.9)
        KEY = ["Classical CNN (small, scratch)", "ResNet50", "EfficientNet-B0", "ConvNeXt-Tiny", "CNN-PCA8 + MLP (matched size)",
               "Hybrid CNN+VQC", "patch-PCA8 + MLP (matched size)", "VQC on patches"]
        for j, k in enumerate([k for k in KEY if k in names]):
            rs = R[k]; y = rs[0]["y_test"]; p = ens(rs, "test"); o = np.argsort(-p)
            tp = np.cumsum(y[o] == 1) / (y == 1).sum(); fp = np.cumsum(y[o] == 0) / (y == 0).sum(); qk = "VQC" in k or "quantum" in k
            ax[0].plot(np.r_[0, fp], np.r_[0, tp], color=PAL[j % len(PAL)], lw=1.8 if qk else 1.0, ls="-" if qk else "--",
                       label=f"{k} ({O['arms'][k]['auc']:.3f})")
        ax[0].plot([0, 1], [0, 1], color="#999", lw=0.8); ax[0].set_xlabel("false-positive rate"); ax[0].set_ylabel("true-positive rate")
        ax[0].set_title(f"ROC, {DSN[ds]} test (ensembles)", loc="left"); ax[0].legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=2)
        au = [O["arms"][k]["auc"] for k in names]
        ax[1].barh(range(len(names)), au, color=[PAL[4] if ("VQC" in k or "quantum" in k) else PAL[0] for k in names])
        ax[1].set_yticks(range(len(names)), names, fontsize=6.5); ax[1].set_xlim(0.45, max(au) + 0.06); ax[1].set_xlabel("test AUC")
        ax[1].invert_yaxis(); ax[1].set_title("AUC by model (red = quantum)", loc="left")
        for i, v_ in enumerate(au):
            ax[1].text(v_ + 0.003, i, f"{v_:.3f}", va="center", fontsize=6.5)
        fig.savefig(f"{D}/S12_{ds}_box9_results.png"); plt.close(fig)
    O["meta"] = META
    return O

for ds in ("cmmd", "cbis"):
    OUT[ds] = do_dataset(ds)
L.extend(["", "=" * 120, "OVERALL VERDICT (pre-registered): a question is SUPPORTED only if it PASSES on BOTH datasets"])
for q, txt in (("Q1", "Hybrid CNN+VQC beats the same-size classical head on the same CNN features"),
               ("Q2", "VQC on patches beats the same-size classical head on the same patch features"),
               ("Q3", "Hybrid CNN+VQC beats a fully fine-tuned ConvNeXt-Tiny")):
    r = [OUT[ds]["endpoints"].get(q) for ds in ("cmmd", "cbis")]
    st = ["missing" if x is None else ("PASS" if x["passed"] else "fail") for x in r]
    L.append(f"  {q}: {txt}: CMMD {st[0]} | CBIS-DDSM {st[1]} -> {'SUPPORTED' if st == ['PASS', 'PASS'] else 'NOT supported'}")

# ---- box 2 (patches) and box 5 (circuit), once ----
X = np.load(os.path.join(DATA, "test_cache.npz")); xt, yt = X["x"], X["y"]
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4)); i0 = int(np.where(yt == 0)[0][0]); i1 = int(np.where(yt == 1)[0][0])
for a_, i, t in ((ax[0], i0, "benign"), (ax[1], i1, "malignant")):
    a_.imshow(xt[i], cmap="gray"); a_.set_title(f"CMMD test image ({t})", loc="left")
    for k in range(1, 4):
        a_.axhline(64 * k, color="#eb6834", lw=0.8); a_.axvline(64 * k, color="#eb6834", lw=0.8)
    a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
small = xt[i1].astype(np.float32).reshape(64, 4, 64, 4).mean((1, 3)); ax[2].imshow(small, cmap="gray"); ax[2].grid(False)
ax[2].set_title("64x64 input, 4x4 patches -> 48 numbers -> PCA 8", loc="left"); ax[2].set_xticks([]); ax[2].set_yticks([])
fig.savefig(f"{D}/S12_box2_patches.png"); plt.close(fig)
fig, ax = plt.subplots(figsize=(10, 3.3)); n = 8; ax.set_xlim(-0.5, 11); ax.set_ylim(-0.8, n); ax.axis("off")
for q in range(n):
    ax.plot([0.3, 10.6], [n - 1 - q] * 2, color="#555", lw=0.8); ax.text(-0.3, n - 1 - q, f"q{q} |0>", va="center", ha="right", fontsize=8)
    ax.add_patch(plt.Rectangle((0.5, n - 1.35 - q), 0.9, 0.7, color="#d9e8fb")); ax.text(0.95, n - 1 - q, "RY(x)", ha="center", va="center", fontsize=6.5)
for l in range(4):
    x0 = 2 + 2.1 * l
    for q in range(n):
        ax.add_patch(plt.Rectangle((x0, n - 1.35 - q), 1.0, 0.7, color="#fde4d8")); ax.text(x0 + 0.5, n - 1 - q, "Rx Ry Rz", ha="center", va="center", fontsize=6)
    for q in range(n):
        t = (q + 1) % n; xc = x0 + 1.2 + 0.08 * q
        ax.plot([xc, xc], [n - 1 - q, n - 1 - t], color="#1baf7a", lw=0.8); ax.plot(xc, n - 1 - q, "o", color="#1baf7a", ms=3)
    ax.text(x0 + 0.8, -0.6, f"layer {l + 1}", ha="center", fontsize=8)
for q in range(n):
    ax.text(10.75, n - 1 - q, "<Z>", va="center", fontsize=7)
ax.set_title("Main VQC: angle encoding, 4 layers of trainable Rx-Ry-Rz + CNOT ring, <Z> readout -> dense -> softmax", loc="left")
fig.savefig(f"{D}/S12_box5_circuit.png"); plt.close(fig)

txt = "\n".join(L); print(txt)
open(f"{D}/S12_report.txt", "w").write(txt + "\n"); json.dump(OUT, open(f"{D}/S12_results.json", "w"), indent=1, default=float)
'''}
for n, s in FILES.items():
    Path(f"{WD}/{n}").write_text(s)
def walk_find(name, root="/kaggle/input"):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git", "ckpt", "train_logs", "logs")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)
def need_dir(summary, parent):
    h = [os.path.dirname(p) for p in walk_find(summary) if os.path.basename(os.path.dirname(p)) == parent]
    if not h:
        raise SystemExit(f"!! {parent}/ not found -> Add Input -> the S8 replication notebook output")
    print(f"{parent:14s} -> {h[0]}"); return h[0]
def need_file(name, where):
    h = walk_find(name)
    if not h:
        raise SystemExit(f"!! {name} not found -> Add Input -> {where}")
    print(f"{name:14s} -> {h[0]}"); return h[0]
DATA = {"cmmd": need_dir("cmmd_summary.json", "cmmd_data"), "cbis": need_dir("clean_summary.json", "clean_data")}
CLF = {"cmmd": need_file("cmmd_clf_s0.pt", "the S8 replication notebook output"),
       "cbis": need_file("a6_clf_s0.pt", "the S2 baseline-audit notebook output (the one S8 also used)")}
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU)
# PennyLane into a SEPARATE folder (used only by the cross-check process; the notebook's own packages are untouched)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", PL, "pennylane"], capture_output=True, text=True)
print("PennyLane install:", "ok" if r.returncode == 0 else "FAILED (cross-check will be skipped) " + r.stderr[-300:])
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True); RC = {}
def run_one(name, gpu, argv, extra_env=None):
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["PA_OUT"] = f"{OUT}/tmp"
    env["CUDA_VISIBLE_DEVICES"] = "" if gpu is None else str(gpu if NGPU >= 2 else 0)
    env.update(extra_env or {})
    print(f">>> {name} started on {'CPU' if gpu is None else 'GPU ' + env['CUDA_VISIBLE_DEVICES']}", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush(); print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]}" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)
print("Cell 1 done.")

In [ ]:
# ===================== CELL 2 — smoke test (~8-10 min) =====================
SM = "/kaggle/working/s12_smoke"; shutil.rmtree(SM, ignore_errors=True)
if os.path.isdir(PL):
    run_one("pennylane_check", None, ["s12_plcheck.py", OUT], {"PYTHONPATH": PL})
    if RC["pennylane_check"] != 0:
        raise SystemExit("!! simulator disagrees with PennyLane (or the check crashed) -> send Claude S12/logs/pennylane_check.log")
for ds in ("cmmd", "cbis"):
    run_one(f"smoke_feats_{ds}", 0, ["s12_gpu.py", "--task", "feats", "--ds", ds, "--data", DATA[ds], "--clf", CLF[ds], "--out", SM, "--smoke"])
    run_one(f"smoke_quantum_{ds}", None, ["s12_quantum.py", "--ds", ds, "--data", DATA[ds], "--feats", f"{SM}/cnn_features_{ds}.npz",
                                          "--out", SM, "--seeds", "0", "--smoke"])
run_one("smoke_cnn", 0, ["s12_gpu.py", "--task", "cnn", "--ds", "cmmd", "--data", DATA["cmmd"], "--out", SM, "--seeds", "0", "--smoke"])
ok = all(v == 0 for k, v in RC.items() if k.startswith("smoke")) and len(glob.glob(f"{SM}/pred_cmmd_cnn_*.npz")) == 4 and \
     all(len(glob.glob(f"{SM}/pred_{ds}_*_VQC_s0.npz")) == 5 and len(glob.glob(f"{SM}/pred_{ds}_*qkernel_s0.npz")) == 2 for ds in ("cmmd", "cbis"))
if not ok:
    raise SystemExit("SMOKE TEST FAILED -> send Claude this block + the S12/logs/smoke_*.log files")
shutil.rmtree(SM, ignore_errors=True); print("SMOKE TEST PASSED")

In [ ]:
# ===================== CELL 3 — full run (~2-3 h): features, then CNNs (GPU 0 = CMMD, GPU 1 = CBIS) + quantum (CPU) =====================
t0 = time.time()
for ds in ("cmmd", "cbis"):
    run_one(f"feats_{ds}", 0, ["s12_gpu.py", "--task", "feats", "--ds", ds, "--data", DATA[ds], "--clf", CLF[ds], "--out", OUT])
    if RC[f"feats_{ds}"] != 0:
        raise SystemExit(f"!! feature extraction failed -> send Claude S12/logs/feats_{ds}.log")
jobs = [(f"cnn_{ds}", g, ["s12_gpu.py", "--task", "cnn", "--ds", ds, "--data", DATA[ds], "--out", OUT]) for g, ds in enumerate(("cmmd", "cbis"))]
jobs += [(f"quantum_{ds}", None, ["s12_quantum.py", "--ds", ds, "--data", DATA[ds], "--feats", f"{OUT}/cnn_features_{ds}.npz", "--out", OUT,
                                  "--budget_h", str(QUANTUM_BUDGET_H)]) for ds in ("cmmd", "cbis")]
ths = [threading.Thread(target=run_one, args=j) for j in jobs]
for t in ths: t.start()
for t in ths: t.join()
print(f"all jobs finished in {(time.time() - t0) / 60:.0f} min | return codes {RC}")

In [ ]:
# ===================== CELL 4 — endpoints, tables, figures, zip =====================
r = subprocess.run([sys.executable, "s12_stats.py", OUT, DATA["cmmd"]], cwd=WD, capture_output=True, text=True)
print(r.stdout[-40000:]); print(r.stderr[-3000:] if r.returncode else "")
from IPython.display import Image, display
for f in sorted(glob.glob(f"{OUT}/S12_*.png")):
    print(os.path.basename(f)); display(Image(filename=f, width=950))
json.dump(RC, open(f"{OUT}/return_codes.json", "w"), indent=1)
shutil.rmtree(f"{OUT}/tmp", ignore_errors=True)
z = shutil.make_archive("/kaggle/working/S12_results", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | return codes {RC}")